# RAG 完整流程实战
## 检索增强生成 (Retrieval Augmented Generation)

本 Notebook 演示完整的 RAG 流程：
1. **加载文档** - 从 PDF 文件读取内容
2. **切割成块** - 将文档分割成小块
3. **向量化** - 将文本转换为向量
4. **存入向量数据库** - 使用 ChromaDB 存储
5. **检索** - 根据用户问题找到最相关的内容
6. **生成回答** - 使用大模型基于检索内容回答问题

## 1. 安装必要的库

In [ ]:
# 如果还没安装，取消注释运行
# !pip install chromadb langchain_chroma langchain_community pypdf langchain-text-splitters

## 2. 加载文档

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# 加载 PDF 文件
file_path = "/Users/logicye/Code/ai_learning/assets/pdf/04tool工具的使用.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"文档总页数: {len(docs)}")
print(f"第一页内容预览: {docs[0].page_content[:200]}...")

## 3. 切割文档成块

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # 每个块的大小（字符数）
    chunk_overlap=30,      # 块之间的重叠字符数
    length_function=len,   # 计算长度的函数
    add_start_index=True   # 添加起始索引
)

# 切割文档
chunks = text_splitter.split_documents(docs)

print(f"切割后的块数: {len(chunks)}")
print(f"第一个块的内容:\n{chunks[0].page_content}")

## 4. 向量化并存入向量数据库

In [ ]:
import chromadb
from langchain_chroma import Chroma
from langchain.embeddings import init_embeddings

# 初始化 Embedding 模型（使用 Ollama 的 Qwen Embedding）
emb = init_embeddings(model="ollama:qwen3-embedding:0.6b")

# 创建 Chroma 向量数据库（内存模式）
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=emb,
    collection_name="rag_knowledge_base"
)

print("✓ 向量数据库创建完成！")
print(f"数据库中向量数量: {vectorstore._collection.count()}")

## 5. 检索 - 根据问题查找相关内容

In [ ]:
# 测试检索功能
query = "工具的核心作用是什么？"

# 检索最相关的3个文档块
results = vectorstore.similarity_search(query, k=3)

print(f"问题: {query}\n")
print("="*60)
for i, doc in enumerate(results, 1):
    print(f"\n[相关文档 {i}]:")
    print(doc.page_content[:300])
    print("-"*60)

## 6. 生成回答 - 使用大模型基于检索内容回答问题

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 初始化聊天模型（使用 Ollama 的 Qwen 模型）
llm = init_chat_model(model="ollama:qwen3:8b")

# 创建检索器
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 格式化文档的辅助函数
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 创建 RAG 提示词模板
prompt = ChatPromptTemplate.from_template("""
你是一个智能助手，请根据提供的的上下文信息来回答用户的问题。

上下文信息：
{context}

用户问题：{question}

请根据以上上下文信息，用简洁准确的中文回答用户的问题。
如果上下文中没有相关信息，请说"抱歉，我暂时没有找到相关信息。"
""")

# 构建 RAG 链
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ RAG 链构建完成！")

In [ ]:
# 测试完整的 RAG 流程
test_questions = [
    "工具的核心作用是什么？",
    "Agent 的工作流程是什么？",
    "如何创建一个工具？"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"问题 {i}: {question}")
    print(f"{'='*60}")
    
    # 使用 RAG 链生成回答
    response = rag_chain.invoke(question)
    
    print(f"\n回答:\n{response}")
    print(f"\n{'='*60}")

## 7. 交互式问答

In [ ]:
# 你可以随时输入问题，系统会从知识库中检索并回答
while True:
    user_question = input("\n请输入你的问题（输入 'quit' 退出）: ")
    
    if user_question.lower() in ['quit', 'exit', 'q', '退出']:
        print("再见！")
        break
    
    if not user_question.strip():
        continue
    
    print("\n正在检索知识库...")
    response = rag_chain.invoke(user_question)
    print(f"\n回答:\n{response}")

## 8. 持久化向量数据库（可选）
如果需要保存向量数据库以便下次直接使用，可以持久化到磁盘：

In [ ]:
# 持久化到磁盘
persist_directory = "./chroma_db"

# 创建持久化的向量数据库
vectorstore_persist = Chroma.from_documents(
    documents=chunks,
    embedding=emb,
    persist_directory=persist_directory,
    collection_name="rag_knowledge_base"
)

print(f"✓ 向量数据库已保存到: {persist_directory}")

# 下次可以直接加载已保存的数据库
# vectorstore_loaded = Chroma(
#     persist_directory=persist_directory,
#     embedding_function=emb,
#     collection_name="rag_knowledge_base"
# )

## 总结

本 Notebook 完整实现了 RAG 的6个核心步骤：

1. **加载文档** ✓ - 使用 `PyPDFLoader` 读取 PDF 文件
2. **切割成块** ✓ - 使用 `RecursiveCharacterTextSplitter` 分割文本
3. **向量化** ✓ - 使用 `Qwen Embedding` 将文本转为向量
4. **存入向量数据库** ✓ - 使用 `ChromaDB` 存储向量
5. **检索** ✓ - 使用相似度搜索找到最相关文档块
6. **生成回答** ✓ - 使用 `Qwen` 大模型基于上下文生成回答

### RAG 的优势：
- ✅ 可以基于最新的、特定领域的知识回答问题
- ✅ 减少大模型的幻觉（hallucination）
- ✅ 不需要重新训练大模型
- ✅ 知识库可以随时更新和扩展